# CreditParity – Steps 1 to 3
**Profitable, Fair Lending Decisions**

**Step 1:** Business case and understanding the data
**Step 2:** Exploratory analysis and bias check
**Step 3:** Data preparation and baseline model (logistic regression)

## Step 1 · Business Case

**Decision.** For every loan application, a retail bank decides whether to approve or decline it.

**Target variable.** `Risk` (1 = customer defaults, 0 = customer repays).

**Error costs.** Both errors cost money, but not equally.
- *False negative* (a defaulter is approved) → the bank loses a large part of the loan amount.
- *False positive* (a good customer is declined) → the bank only loses the interest margin.

The classic cost matrix for the German Credit dataset treats a missed default as **5 times as expensive** as a rejected good customer. We will use this assumption later to find the optimal threshold.

**Fairness question.** The dataset is called `biased_training`. We therefore check whether age, sex or origin unfairly influence the decision.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# Load data
# In the repo the file lives in data/. In Google Colab, upload the CSV via the
# folder icon on the left and set DATA_PATH = "german_credit_data_biased_training.csv"
DATA_PATH = "../data/german_credit_data_biased_training.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

In [ ]:
# Encode the target as 0/1
df["y"] = (df["Risk"] == "Risk").astype(int)

print("Missing values:", df.isna().sum().sum())
print("Default rate:", round(df["y"].mean(), 3))
df["Risk"].value_counts()

**What we see.** 5,000 applications, 20 features, no missing values. About one third of customers default. The classes are moderately imbalanced, so accuracy alone is not a sufficient metric.

## Step 2 · Exploratory Analysis and Bias Check

In [ ]:
# Default rate by the most important categorical features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["CheckingStatus", "CreditHistory", "ExistingSavings"]):
    df.groupby(col)["y"].mean().sort_values().plot.barh(ax=ax, color="#4C72B0")
    ax.set_title(f"Default rate by {col}")
    ax.set_xlabel("Share of defaults")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric features: distribution by risk class
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["LoanAmount", "LoanDuration", "Age"]):
    sns.boxplot(data=df, x="Risk", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Bias check: default rate by sensitive attributes
df["AgeGroup"] = pd.cut(df["Age"], bins=[18, 25, 35, 50, 80],
                        labels=["19-25", "26-35", "36-50", "51+"])

for col in ["Sex", "ForeignWorker", "AgeGroup"]:
    print(df.groupby(col, observed=True)["y"].agg(default_rate="mean", count="count").round(3), "\n")

**Key observation.** Age is extremely strongly related to risk. About 7% of 19 to 25-year-olds default, compared with almost 78% of those over 50. Foreign workers also show a clearly higher default rate, but this group has only 123 cases.

This is a warning sign. A model that learns these patterns would systematically reject older applicants. In the EU this would be legally problematic, since the EU AI Act classifies credit scoring as a high-risk AI system and anti-discrimination law protects age and sex. This tension becomes the core of our story: a model that is profitable **and** fair.

Some relationships are also counter-intuitive (e.g. `less_0` on the checking account has the lowest default rate). The dataset is synthetic, which we should state openly as a limitation in the presentation.

## Step 3 · Data Preparation and Baseline Model

In [ ]:
# Separate features and target
SENSITIVE = ["Sex", "Age", "ForeignWorker"]   # used for the fairness comparison later
X = df.drop(columns=["Risk", "y", "AgeGroup"])
y = df["y"]

categorical = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric = X.select_dtypes(include="number").columns.tolist()
print("Categorical:", len(categorical), "| Numeric:", len(numeric))

# Stratified split: 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
# Preprocessing: one-hot encoding for categories, standardization for numbers
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric),
])

# Baseline 0: naive model that always predicts the most frequent class
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

# Baseline 1: logistic regression
logreg = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)

In [ ]:
# Evaluate both models on the test set
def evaluate(name, model, X_t, y_t, threshold=0.5):
    proba = model.predict_proba(X_t)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_t, pred),
        "Precision": precision_score(y_t, pred, zero_division=0),
        "Recall": recall_score(y_t, pred),
        "F1": f1_score(y_t, pred),
        "ROC AUC": roc_auc_score(y_t, proba),
    }

results = pd.DataFrame([
    evaluate("Dummy (always 'No Risk')", dummy, X_test, y_test),
    evaluate("Logistic regression", logreg, X_test, y_test),
]).set_index("Model").round(3)
results

In [ ]:
# Confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_estimator(
    logreg, X_test, y_test, display_labels=["No Risk", "Risk"],
    cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion matrix (threshold 0.5)")
RocCurveDisplay.from_estimator(logreg, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()

In [ ]:
# What drives the model? The 15 largest coefficients
names = logreg.named_steps["prep"].get_feature_names_out()
coef = pd.Series(logreg.named_steps["model"].coef_[0], index=names)
top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(15)

top.sort_values().plot.barh(figsize=(8, 6),
    color=["#C44E52" if v > 0 else "#55A868" for v in top.sort_values()])
plt.title("Top 15 drivers (red = increases risk, green = decreases risk)")
plt.tight_layout()
plt.show()

### Benchmark: IBM watsonx AutoAI

As a starting point, we ran the same dataset through **IBM watsonx AutoAI**, which automatically tests several pipelines and ranks them. AutoAI selected an **XGBoost classifier** as the best model, ranked by accuracy on a 10% holdout set (499 applications).

| | AutoAI XGBoost (holdout, 499 cases) | Our logistic regression (test set, 1,000 cases) |
|---|---|---|
| Accuracy | 79.6% | 80.0% |
| Recall (defaults caught) | 53.3% | 62.6% |
| Precision (flagged cases that really default) | 78.8% | 73.6% |
| Most important features | LoanAmount 20%, LoanDuration 17%, Age 14% | CreditHistory, OwnsProperty, EmploymentDuration |

At first glance our model catches more defaults, but it also raises more false alarms. The two models simply sit at different points of the same precision–recall trade-off, and they were evaluated on different test sets. To compare them fairly, we evaluate both model types on **the same data** with **5-fold cross-validation** and a threshold-independent metric (ROC AUC).

In [ ]:
# Fair comparison: same data, same folds, threshold-independent metric
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

# Gradient-boosted trees, the same model family as XGBoost
gboost = Pipeline([
    ("prep", ColumnTransformer([("cat", OrdinalEncoder(), categorical)],
                               remainder="passthrough")),
    ("model", HistGradientBoostingClassifier(
        categorical_features=list(range(len(categorical))),
        random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in [("Logistic regression", logreg), ("Gradient boosting (XGBoost-type)", gboost)]:
    scores = cross_validate(model, X, y, cv=cv, scoring=["roc_auc", "accuracy"])
    rows.append({"Model": name,
                 "ROC AUC (mean)": scores["test_roc_auc"].mean(),
                 "ROC AUC (std)": scores["test_roc_auc"].std(),
                 "Accuracy (mean)": scores["test_accuracy"].mean()})
cv_results = pd.DataFrame(rows).set_index("Model").round(3)
cv_results

**Result.** On the same data, the simple logistic regression performs **just as well** as the gradient-boosted trees selected by AutoAI. The difference in ROC AUC is smaller than the variation between folds.

**What this means for CreditParity.**
- **We keep the model simple.** A logistic regression is transparent: every coefficient can be explained to a credit analyst, a customer or a regulator. Under the EU AI Act, credit scoring is a high-risk use case, so explainability is a real advantage when performance is equal.
- **The real lever is the decision, not the algorithm.** AutoAI optimized for accuracy, but for a bank a missed default is far more expensive than a rejected good customer. Instead of a more complex model, we focus on **catching more defaults** by setting the threshold according to the bank's costs.
- **Machines and humans work together.** Clear cases are decided automatically, while uncertain cases go to a credit analyst. This human-in-the-loop policy keeps costs low without leaving borderline decisions entirely to the model.

**Summary of Step 3.** The logistic regression clearly beats the dummy model. Important for the presentation: the dummy model already reaches about 67% accuracy without detecting a single default (recall = 0). That is why we evaluate with recall, precision and ROC AUC.

**Two findings for the fairness part.**
- *Sex.* `Sex_male` is among the top 15 drivers and lowers the predicted risk. Women have the **lower** raw default rate (30% vs. 36%), yet holding all other features equal, the model rates them as riskier. This is the built-in bias.
- *Age.* Although age was extremely strongly related to risk in Step 2, `Age` has almost no weight in the logistic regression (coefficient ≈ 0.03). The effect runs through other features, especially `EmploymentDuration_greater_7`, which act as **proxy variables**. Tree-based models such as XGBoost use `Age` directly instead. It follows that simply dropping sensitive features does not reliably remove bias. We test this in Step 6.

**Next steps.**
4. Confirm the model comparison with XGBoost and a random forest (cross-validation)
5. Optimize the threshold based on the 5:1 cost matrix and develop a three-zone human-in-the-loop policy (auto-approve, manual review, decline recommended)
6. Measure fairness and compare a model without sensitive features
7. Explainability per customer (SHAP)
8. Dashboard with Streamlit